# Student Admission Fairness Analysis

## Overview
This notebook explores fairness in student admission decisions using a synthetic dataset. We examine how machine learning models perform when making admission predictions and analyze potential biases across different demographic groups.

## Research Questions
1. Can AIDE-ML create an accurate admission prediction model?
2. Does the baseline model exhibit fairness issues across protected attributes?
3. What methods can improve fairness while maintaining accuracy?

## Approach
- **Baseline**: Simple AIDE prompt for admission prediction
- **Improvements**: Pre-processing, post-processing, fairness-aware prompting, custom pipelines

## 1. Generate Synthetic Student Admission Dataset

We create a realistic synthetic dataset with:
- **10,000+ student applications**
- **Academic features**: GPA, SAT/ACT scores, AP courses, extracurriculars
- **Demographic features**: Gender, race/ethnicity, socioeconomic status, geography
- **Admission outcome**: Binary (Admitted/Rejected)
- **Built-in biases**: Subtle discrimination patterns to test fairness metrics

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set random seed for reproducibility
np.random.seed(42)

# Dataset size
n_samples = 12000

print(f"Generating {n_samples:,} synthetic student applications...")

Generating 12,000 synthetic student applications...


### 1.1 Define Dataset Attributes

#### Academic Features (Merit-based)
- **gpa**: High school GPA (0.0 - 4.0)
- **sat_score**: SAT score (400 - 1600)
- **act_score**: ACT score (1 - 36)
- **ap_courses**: Number of Advanced Placement courses taken (0 - 12)
- **honors_courses**: Number of honors courses (0 - 15)
- **class_rank_percentile**: Class rank as percentile (0 - 100)
- **extracurriculars**: Number of extracurricular activities (0 - 10)
- **leadership_positions**: Number of leadership roles (0 - 5)
- **community_service_hours**: Volunteer hours (0 - 500)
- **awards_honors**: Number of academic awards (0 - 8)

#### Essay & Recommendations
- **essay_score**: Essay quality score (1 - 10)
- **recommendation_strength**: Letter of recommendation strength (1 - 10)

#### Protected/Demographic Attributes (Fairness-sensitive)
- **gender**: Male, Female, Non-binary
- **race**: White, Black, Hispanic, Asian, Native American, Other
- **socioeconomic_status**: Low, Middle, High
- **first_generation**: Whether student is first-generation college student (Yes/No)
- **region**: Geographic region (Northeast, Southeast, Midwest, Southwest, West)
- **urban_rural**: Urban, Suburban, Rural
- **disability_status**: Has documented disability (Yes/No)

#### Target Variable
- **admitted**: 1 (Admitted) or 0 (Rejected)

In [2]:
# Generate protected attributes first (these will influence other features due to systemic biases)

# Gender distribution
gender = np.random.choice(['Male', 'Female', 'Non-binary'], 
                          size=n_samples, 
                          p=[0.48, 0.48, 0.04])

# Race/ethnicity distribution (roughly based on US demographics)
race = np.random.choice(['White', 'Black', 'Hispanic', 'Asian', 'Native American', 'Other'],
                        size=n_samples,
                        p=[0.55, 0.13, 0.18, 0.10, 0.02, 0.02])

# Socioeconomic status (correlated with race - modeling systemic inequality)
socioeconomic_status = []
for r in race:
    if r == 'White':
        ses = np.random.choice(['Low', 'Middle', 'High'], p=[0.25, 0.45, 0.30])
    elif r == 'Asian':
        ses = np.random.choice(['Low', 'Middle', 'High'], p=[0.20, 0.40, 0.40])
    elif r in ['Black', 'Hispanic', 'Native American']:
        ses = np.random.choice(['Low', 'Middle', 'High'], p=[0.45, 0.40, 0.15])
    else:
        ses = np.random.choice(['Low', 'Middle', 'High'], p=[0.33, 0.34, 0.33])
    socioeconomic_status.append(ses)

socioeconomic_status = np.array(socioeconomic_status)

# First generation college student (correlated with SES)
first_generation = []
for ses in socioeconomic_status:
    if ses == 'Low':
        fg = np.random.choice(['Yes', 'No'], p=[0.65, 0.35])
    elif ses == 'Middle':
        fg = np.random.choice(['Yes', 'No'], p=[0.35, 0.65])
    else:  # High
        fg = np.random.choice(['Yes', 'No'], p=[0.10, 0.90])
    first_generation.append(fg)

first_generation = np.array(first_generation)

# Geographic region
region = np.random.choice(['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West'],
                         size=n_samples,
                         p=[0.22, 0.20, 0.18, 0.18, 0.22])

# Urban/Suburban/Rural (correlated with SES)
urban_rural = []
for ses in socioeconomic_status:
    if ses == 'High':
        ur = np.random.choice(['Urban', 'Suburban', 'Rural'], p=[0.35, 0.55, 0.10])
    elif ses == 'Middle':
        ur = np.random.choice(['Urban', 'Suburban', 'Rural'], p=[0.30, 0.50, 0.20])
    else:  # Low
        ur = np.random.choice(['Urban', 'Suburban', 'Rural'], p=[0.40, 0.30, 0.30])
    urban_rural.append(ur)

urban_rural = np.array(urban_rural)

# Disability status
disability_status = np.random.choice(['Yes', 'No'], 
                                    size=n_samples, 
                                    p=[0.12, 0.88])

print("Generated protected attributes with realistic correlations")

Generated protected attributes with realistic correlations


In [ ]:
# Generate academic features (influenced by socioeconomic factors - modeling educational inequality)

def generate_gpa(ses, race):
    """Generate GPA with bias towards higher SES and certain races"""
    base_mean = 3.2
    
    # SES adjustment (resource availability)
    if ses == 'High':
        mean_adj = 0.25
    elif ses == 'Middle':
        mean_adj = 0.10
    else:
        mean_adj = -0.10
    
    # Small race adjustment (modeling systemic educational disparities)
    if race in ['Asian', 'White']:
        race_adj = 0.08
    else:
        race_adj = -0.05
    
    gpa = np.clip(np.random.normal(base_mean + mean_adj + race_adj, 0.4), 0.0, 4.0)
    return round(gpa, 2)

def generate_sat(ses, race, urban_rural):
    """Generate SAT score with systemic biases"""
    base_mean = 1050
    
    # Strong SES correlation (test prep access)
    if ses == 'High':
        mean_adj = 150
    elif ses == 'Middle':
        mean_adj = 50
    else:
        mean_adj = -80
    
    # Race adjustment (modeling test bias)
    if race == 'Asian':
        race_adj = 80
    elif race == 'White':
        race_adj = 40
    else:
        race_adj = -30
    
    # Urban/suburban advantage (resource access)
    if urban_rural == 'Suburban':
        ur_adj = 30
    elif urban_rural == 'Urban':
        ur_adj = 10
    else:
        ur_adj = -20
    
    sat = np.clip(np.random.normal(base_mean + mean_adj + race_adj + ur_adj, 150), 400, 1600)
    return int(round(sat / 10) * 10)  # Round to nearest 10

def generate_act(sat_score):
    """Convert SAT to ACT (roughly correlated)"""
    # Approximate SAT to ACT conversion
    act = (sat_score - 400) / 1200 * 35 + 1
    act += np.random.normal(0, 2)  # Add some noise
    return int(np.clip(act, 1, 36))

# Generate academic metrics
gpa = np.array([generate_gpa(ses, r) for ses, r in zip(socioeconomic_status, race)])
sat_score = np.array([generate_sat(ses, r, ur) for ses, r, ur in zip(socioeconomic_status, race, urban_rural)])
act_score = np.array([generate_act(sat) for sat in sat_score])

# AP courses (wealth dependent)
ap_courses = []
for ses in socioeconomic_status:
    if ses == 'High':
        ap = int(np.clip(np.random.normal(6, 2), 0, 12))
    elif ses == 'Middle':
        ap = int(np.clip(np.random.normal(3, 2), 0, 12))
    else:
        ap = int(np.clip(np.random.normal(1, 1.5), 0, 12))
    ap_courses.append(ap)

ap_courses = np.array(ap_courses)

# Honors courses
honors_courses = np.array([int(np.clip(np.random.normal(ap * 1.2 + 2, 2), 0, 15)) for ap in ap_courses])

# Class rank (correlated with GPA)
class_rank_percentile = np.array([int(np.clip((g / 4.0) * 100 + np.random.normal(0, 10), 0, 100)) for g in gpa])

# Extracurriculars (wealth and time dependent)
extracurriculars = []
for ses in socioeconomic_status:
    if ses == 'High':
        extra = int(np.clip(np.random.normal(5, 2), 0, 10))
    elif ses == 'Middle':
        extra = int(np.clip(np.random.normal(3, 1.5), 0, 10))
    else:
        extra = int(np.clip(np.random.normal(1.5, 1), 0, 10))  # Low SES students often work
    extracurriculars.append(extra)

extracurriculars = np.array(extracurriculars)

# Leadership positions
leadership_positions = np.array([int(np.clip(np.random.poisson(ex / 2.5), 0, 5)) for ex in extracurriculars])

# Community service hours
community_service_hours = np.array([int(np.clip(np.random.gamma(2, 30), 0, 500)) for _ in range(n_samples)])

# Academic awards (ensure lambda is non-negative for Poisson distribution)
awards_honors = np.array([int(np.clip(np.random.poisson(max(0, (g - 2.5) * 2)), 0, 8)) for g in gpa])

# Essay score (slight gender bias - females tend to score higher on writing)
essay_score = []
for g in gender:
    if g == 'Female':
        score = np.clip(np.random.normal(6.5, 1.5), 1, 10)
    else:
        score = np.clip(np.random.normal(6.0, 1.5), 1, 10)
    essay_score.append(round(score, 1))

essay_score = np.array(essay_score)

# Recommendation strength (correlated with class rank and SES - access to influential recommenders)
recommendation_strength = []
for rank, ses in zip(class_rank_percentile, socioeconomic_status):
    base = rank / 100 * 8 + 2
    if ses == 'High':
        base += 0.5  # Better access to strong recommenders
    rec = np.clip(np.random.normal(base, 1), 1, 10)
    recommendation_strength.append(round(rec, 1))

recommendation_strength = np.array(recommendation_strength)

print("Generated academic features with realistic socioeconomic correlations")

ValueError: lam < 0 or lam is NaN

In [ ]:
# Generate admission decisions (with built-in biases)

def calculate_admission_probability(row_idx):
    """
    Calculate admission probability based on features.
    Includes subtle biases to test fairness metrics.
    """
    # Academic merit score (normalized 0-1)
    academic_score = (
        (gpa[row_idx] / 4.0) * 0.25 +
        (sat_score[row_idx] / 1600) * 0.20 +
        (act_score[row_idx] / 36) * 0.15 +
        (ap_courses[row_idx] / 12) * 0.10 +
        (class_rank_percentile[row_idx] / 100) * 0.15 +
        (extracurriculars[row_idx] / 10) * 0.05 +
        (essay_score[row_idx] / 10) * 0.05 +
        (recommendation_strength[row_idx] / 10) * 0.05
    )
    
    # Base probability from academic merit
    prob = academic_score
    
    # BIAS 1: Subtle gender bias (historical bias in STEM admission)
    if gender[row_idx] == 'Male':
        prob += 0.02
    elif gender[row_idx] == 'Female':
        prob += 0.01
    
    # BIAS 2: Race bias (modeling historical discrimination)
    if race[row_idx] in ['White', 'Asian']:
        prob += 0.03
    elif race[row_idx] in ['Black', 'Hispanic', 'Native American']:
        prob -= 0.02  # Systemic disadvantage despite equal merit
    
    # BIAS 3: SES bias (wealthy students have advantages beyond academics)
    if socioeconomic_status[row_idx] == 'High':
        prob += 0.04  # Legacy, donations, connections
    elif socioeconomic_status[row_idx] == 'Low':
        prob -= 0.02
    
    # BIAS 4: Geographic bias (regional diversity goals)
    if region[row_idx] in ['Northeast', 'West']:
        prob += 0.01  # Competitive regions
    
    # BIAS 5: First-generation small penalty (despite diversity claims)
    if first_generation[row_idx] == 'Yes':
        prob -= 0.01
    
    # Add some randomness
    prob += np.random.normal(0, 0.05)
    
    # Clip to valid probability range
    return np.clip(prob, 0, 1)

# Calculate probabilities and make decisions
admission_probabilities = np.array([calculate_admission_probability(i) for i in range(n_samples)])

# Set admission threshold to get ~30% admission rate
threshold = np.percentile(admission_probabilities, 70)
admitted = (admission_probabilities > threshold).astype(int)

admission_rate = admitted.mean()
print(f"\nOverall admission rate: {admission_rate:.1%}")
print(f"Admission threshold: {threshold:.3f}")

In [ ]:
# Create DataFrame
df = pd.DataFrame({
    # Demographics (Protected Attributes)
    'gender': gender,
    'race': race,
    'socioeconomic_status': socioeconomic_status,
    'first_generation': first_generation,
    'region': region,
    'urban_rural': urban_rural,
    'disability_status': disability_status,
    
    # Academic Metrics
    'gpa': gpa,
    'sat_score': sat_score,
    'act_score': act_score,
    'ap_courses': ap_courses,
    'honors_courses': honors_courses,
    'class_rank_percentile': class_rank_percentile,
    
    # Activities
    'extracurriculars': extracurriculars,
    'leadership_positions': leadership_positions,
    'community_service_hours': community_service_hours,
    'awards_honors': awards_honors,
    
    # Application Materials
    'essay_score': essay_score,
    'recommendation_strength': recommendation_strength,
    
    # Target
    'admitted': admitted
})

print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head(10)

## 2. Dataset Statistics and Exploration

In [ ]:
# Basic statistics
print("Dataset Information:")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("Numerical Features Statistics:")
print("=" * 60)
df.describe()

In [ ]:
# Categorical features distribution
print("\nCategorical Features Distribution:")
print("=" * 60)

categorical_cols = ['gender', 'race', 'socioeconomic_status', 'first_generation', 
                    'region', 'urban_rural', 'disability_status', 'admitted']

for col in categorical_cols:
    print(f"\n{col.upper().replace('_', ' ')}:")
    counts = df[col].value_counts()
    percentages = df[col].value_counts(normalize=True) * 100
    
    for idx in counts.index:
        print(f"  {idx}: {counts[idx]:,} ({percentages[idx]:.1f}%)")

## 3. Fairness Analysis - Admission Rates by Protected Groups

In [ ]:
# Analyze admission rates across protected groups
print("\nADMISSION RATES BY PROTECTED ATTRIBUTES")
print("=" * 60)

def analyze_group_rates(column_name):
    """Calculate and display admission rates for each group"""
    print(f"\n{column_name.upper().replace('_', ' ')}:")
    rates = df.groupby(column_name)['admitted'].agg(['sum', 'count', 'mean'])
    rates.columns = ['Admitted', 'Total', 'Rate']
    rates['Rate'] = rates['Rate'] * 100
    rates = rates.sort_values('Rate', ascending=False)
    print(rates.to_string())
    
    # Calculate disparity
    max_rate = rates['Rate'].max()
    min_rate = rates['Rate'].min()
    disparity = max_rate - min_rate
    print(f"  → Disparity: {disparity:.2f} percentage points")
    return rates

# Analyze each protected attribute
protected_attrs = ['gender', 'race', 'socioeconomic_status', 'first_generation', 
                   'region', 'urban_rural', 'disability_status']

for attr in protected_attrs:
    analyze_group_rates(attr)

In [ ]:
# Visualize admission rates
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for idx, attr in enumerate(protected_attrs):
    rates = df.groupby(attr)['admitted'].mean() * 100
    rates = rates.sort_values(ascending=False)
    
    ax = axes[idx]
    bars = ax.bar(range(len(rates)), rates.values, color='steelblue', alpha=0.7)
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels(rates.index, rotation=45, ha='right')
    ax.set_ylabel('Admission Rate (%)', fontsize=10)
    ax.set_title(f'Admission Rate by {attr.replace("_", " ").title()}', fontsize=11, fontweight='bold')
    ax.axhline(y=df['admitted'].mean() * 100, color='red', linestyle='--', 
               linewidth=1, alpha=0.7, label='Overall Rate')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    
    # Annotate bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%',
                ha='center', va='bottom', fontsize=8)

# Remove extra subplots
for idx in range(len(protected_attrs), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('admission_rates_by_group.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'admission_rates_by_group.png'")

## 4. Academic Performance by Demographics

Examine whether academic metrics differ across groups (revealing systemic inequalities)

In [ ]:
# Compare average academic metrics across groups
print("\nAVERAGE ACADEMIC METRICS BY RACE")
print("=" * 80)
academic_by_race = df.groupby('race')[['gpa', 'sat_score', 'act_score', 'ap_courses']].mean()
academic_by_race = academic_by_race.round(2)
print(academic_by_race.to_string())

print("\nAVERAGE ACADEMIC METRICS BY SOCIOECONOMIC STATUS")
print("=" * 80)
academic_by_ses = df.groupby('socioeconomic_status')[['gpa', 'sat_score', 'act_score', 'ap_courses']].mean()
academic_by_ses = academic_by_ses.round(2)
print(academic_by_ses.to_string())

In [ ]:
# Visualize GPA and SAT distributions by protected groups
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# GPA by Race
df.boxplot(column='gpa', by='race', ax=axes[0, 0])
axes[0, 0].set_title('GPA Distribution by Race', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Race')
axes[0, 0].set_ylabel('GPA')
plt.sca(axes[0, 0])
plt.xticks(rotation=45, ha='right')

# SAT by Race
df.boxplot(column='sat_score', by='race', ax=axes[0, 1])
axes[0, 1].set_title('SAT Score Distribution by Race', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Race')
axes[0, 1].set_ylabel('SAT Score')
plt.sca(axes[0, 1])
plt.xticks(rotation=45, ha='right')

# GPA by SES
df.boxplot(column='gpa', by='socioeconomic_status', ax=axes[1, 0])
axes[1, 0].set_title('GPA Distribution by Socioeconomic Status', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Socioeconomic Status')
axes[1, 0].set_ylabel('GPA')

# SAT by SES
df.boxplot(column='sat_score', by='socioeconomic_status', ax=axes[1, 1])
axes[1, 1].set_title('SAT Score Distribution by Socioeconomic Status', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Socioeconomic Status')
axes[1, 1].set_ylabel('SAT Score')

plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.savefig('academic_metrics_by_demographics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'academic_metrics_by_demographics.png'")

## 5. Save Dataset

Save the synthetic dataset for use with AIDE-ML

In [ ]:
# Create output directory
output_dir = Path('../../resources/datasets/student-admission')
output_dir.mkdir(parents=True, exist_ok=True)

# Save full dataset
full_path = output_dir / 'student_admission_full.csv'
df.to_csv(full_path, index=False)
print(f"Saved full dataset: {full_path}")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)}")

# Create train/test split (80/20)
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['admitted'])

train_path = output_dir / 'train.csv'
test_path = output_dir / 'test.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"\nSaved train set: {train_path}")
print(f"  Rows: {len(train_df):,}")
print(f"  Admission rate: {train_df['admitted'].mean():.1%}")

print(f"\nSaved test set: {test_path}")
print(f"  Rows: {len(test_df):,}")
print(f"  Admission rate: {test_df['admitted'].mean():.1%}")

print("\n" + "=" * 60)
print("Dataset generation complete!")
print("=" * 60)

## 6. Dataset Summary

### Key Characteristics:

1. **Size**: 12,000 student applications
2. **Admission Rate**: ~30%
3. **Features**: 20 features (7 protected attributes + 13 merit-based)

### Built-in Fairness Issues:

The dataset contains realistic systemic biases:

1. **Socioeconomic Disparities**:
   - Higher SES → Better SAT scores (test prep access)
   - Higher SES → More AP courses (school resources)
   - Higher SES → Stronger recommendations (network access)

2. **Racial Disparities**:
   - Correlation between race and SES (historical inequalities)
   - Test score gaps across racial groups
   - Admission bias favoring White and Asian applicants

3. **Gender Bias**:
   - Slight admission preference for male applicants
   - Gender differences in essay scores

4. **Geographic Disparities**:
   - Rural students have fewer resources
   - Regional admission rate variations

5. **First-Generation Penalty**:
   - Despite diversity rhetoric, first-gen students face disadvantages

### Fairness Metrics to Evaluate:

- **Demographic Parity**: Equal admission rates across groups
- **Equalized Odds**: Equal TPR/FPR across groups
- **Equal Opportunity**: Equal TPR for qualified applicants
- **Predictive Parity**: Equal precision across groups
- **Calibration**: Predicted probabilities match actual outcomes across groups

### Next Steps:

1. Run AIDE-ML baseline model
2. Evaluate fairness metrics
3. Implement fairness improvements
4. Compare results